# 04o2: Loading Vector Embeddings into Qdrant

This notebook demonstrates how to load vector embeddings into Qdrant for semantic search.

## Prerequisites

**⚠️ Important:** Before running this notebook, ensure you have:
- Completed [**00-import.ipynb**](./00-import.ipynb) for environment detection and connection setup
- Completed [**02-extracting-embeddings.ipynb**](./02-extracting-embeddings.ipynb) to generate embeddings first
- Qdrant service is running and accessible

All environment detection, Qdrant connection, and configuration are handled in `00-import.ipynb`.

## Overview

This notebook stores vector embeddings in Qdrant for semantic search. It takes embeddings generated in [**02-extracting-embeddings.ipynb**](./02-extracting-embeddings.ipynb) and stores them in Qdrant's vector collection, enabling semantic search capabilities on your knowledge graph.

**Alternative Approach:** If you prefer storing embeddings in Neo4j instead of Qdrant, see [**04o1-loading-vector-embeddings-neo4j.ipynb**](./04o1-loading-vector-embeddings-neo4j.ipynb) for an alternative vector store.

We'll:
1. Load embeddings generated from [**02-extracting-embeddings.ipynb**](./02-extracting-embeddings.ipynb)
2. Ensure Qdrant collection exists
3. Store notes with embeddings in Qdrant vector store

**Note:** Embeddings must be generated in [**02-extracting-embeddings.ipynb**](./02-extracting-embeddings.ipynb) first.


In [ ]:
# Run common imports and setup
%run 00-import.ipynb

# Additional imports specific to this notebook
import hashlib
from qdrant_client.models import PointStruct

print("✅ Additional libraries imported")
print("⚠️  Note: This notebook expects embeddings to be generated in 02-extracting-embeddings.ipynb")
print("   If running in the same kernel session, embeddings, file_contents, and file_metadata")
print("   should be available from that notebook.")


In [ ]:
# All setup is done in 00-import.ipynb - no additional configuration needed here


## Load Pre-Generated Embeddings

This notebook expects embeddings to be generated in [**02-extracting-embeddings.ipynb**](./02-extracting-embeddings.ipynb). 

**If running in the same kernel session:** The variables `embeddings`, `file_contents`, and `file_metadata` from notebook 02 should be available here.

**If running in a new kernel session:** You'll need to re-run notebook 02 first, or load saved embeddings.


In [ ]:
# Check if embeddings are available from notebook 02-extracting-embeddings.ipynb
try:
    # Check if variables exist from previous notebook
    if 'embeddings' not in globals() or 'file_contents' not in globals() or 'file_metadata' not in globals():
        raise NameError("Embeddings not found")
    
    print(f"✅ Found embeddings from notebook 02-extracting-embeddings.ipynb")
    print(f"   Number of embeddings: {len(embeddings)}")
    print(f"   Number of files: {len(file_contents)}")
    print(f"   Embedding dimension: {len(embeddings[0]) if embeddings else 0}")
except NameError:
    print("❌ Error: Embeddings not found!")
    print("   Please run 02-extracting-embeddings.ipynb first in the same kernel session,")
    print("   or re-generate embeddings by running that notebook.")
    raise RuntimeError("Embeddings must be generated in 02-extracting-embeddings.ipynb before running this notebook.")


In [ ]:
# Verify embeddings match file contents
if len(embeddings) != len(file_contents):
    raise ValueError(
        f"Mismatch: {len(embeddings)} embeddings but {len(file_contents)} files. "
        "Make sure embeddings and files are from the same run of 02-extracting-embeddings.ipynb"
    )

print(f"✅ Verified: {len(embeddings)} embeddings match {len(file_contents)} files")


## Ensure Qdrant Collection Exists

Create the Qdrant collection if it doesn't exist. The collection will be configured with the correct vector size based on your embedding model.


In [ ]:
# Get Qdrant client from dependencies
qdrant_client = dependencies.vector_store_client_manager.get_client()

# Ensure collection exists with correct vector size
collection_name = settings.qdrant_collection_name
vector_size = settings.get_embedding_size(settings.litellm_proxy_embedding_model)

print(f"Collection name: {collection_name}")
print(f"Vector size: {vector_size}")

# Create collection if it doesn't exist
dependencies.vector_store_client_manager.ensure_collection(
    collection_name=collection_name,
    vector_size=vector_size,
    recreate=False,  # Don't recreate if exists
)

print(f"✅ Collection '{collection_name}' is ready")


## Store Notes with Embeddings

Store the notes and their embeddings in Qdrant as points.


In [ ]:
# Create points for Qdrant
# Each point contains: id (hashed file path), vector (embedding), payload (metadata)
points = []
for idx, (embedding, metadata) in enumerate(zip(embeddings, file_metadata)):
    # Use file path as ID (hash it for uniqueness)
    point_id = hashlib.md5(metadata["file_path"].encode()).hexdigest()
    
    # Add file content to payload for retrieval
    payload = metadata.copy()
    payload["content"] = file_contents[idx]
    
    points.append(
        PointStruct(
            id=point_id,
            vector=embedding,
            payload=payload,
        )
    )
    
    # Print progress
    if (idx + 1) % 10 == 0:
        print(f"  Prepared {idx + 1}/{len(embeddings)} points...")

print(f"✅ Prepared {len(points)} points for Qdrant")


In [ ]:
# Upsert points to Qdrant
print(f"Storing {len(points)} points in Qdrant collection '{collection_name}'...")
qdrant_client.upsert(collection_name=collection_name, points=points)
print(f"✅ Successfully stored {len(points)} points in Qdrant")


## Verify Data

Check how many points are stored in the Qdrant collection.


In [ ]:
# Get collection info
collection_info = qdrant_client.get_collection(collection_name)
print(f"✅ Collection '{collection_name}' info:")
print(f"   Points count: {collection_info.points_count}")
print(f"   Vector size: {collection_info.config.params.vectors.size}")
print(f"   Distance metric: {collection_info.config.params.vectors.distance}")


## Next Steps

Now that vector embeddings are loaded into Qdrant, proceed to:
- [**05-querying-graph.ipynb**](./05-querying-graph.ipynb): Query the graph with graph patterns
- [**06-querying-vector-embeddings.ipynb**](./06-querying-vector-embeddings.ipynb): Query using vector similarity search (works with both Neo4j and Qdrant)
